# CoLM — LLMs with Notebooks

Run this notebook in Google Colab to start CoLM. You'll get:
- A Jupyter-like notebook web UI
- An AI assistant sidebar with tool use (code execution, cell management, file browsing)
- A public URL via ngrok

**Requirements:**
- An [OpenRouter API key](https://openrouter.ai/keys) (free tier available)
- An [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken) (free tier available)

In [ ]:
# Install Node.js 20+ if not already present
import subprocess, sys, os

try:
    result = subprocess.run(['node', '--version'], capture_output=True, text=True, check=True)
    version = result.stdout.strip()
    print(f'Node.js already installed: {version}')
    major = int(version.lstrip('v').split('.')[0])
    if major < 18:
        print('Version too old, upgrading...')
        raise Exception('old version')
except:
    print('Installing Node.js 20...')
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    !node --version

In [ ]:
# Clone the CoLM repository
import os
if not os.path.isdir('colm'):
    !git clone https://github.com/reecdev/colm.git
%cd colm

In [ ]:
# Install npm dependencies and build the frontend
!npm install
!npm run build

In [ ]:
# Install ngrok
!pip install -q pyngrok
!which ngrok || (curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null && echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | tee /etc/apt/sources.list.d/ngrok.list >/dev/null && apt-get update -qq && apt-get install -y -qq ngrok)

In [ ]:
# Set your API keys — you'll be prompted for these
from google.colab import userdataimport getpass

openrouter_key = os.environ.get('OPENROUTER_API_KEY') or userdata.get('OPENROUTER_API_KEY') or getpass.getpass('OpenRouter API key: ')
ngrok_token = os.environ.get('NGROK_TOKEN') or userdata.get('NGROK_TOKEN') or getpass.getpass('ngrok auth token (press Enter to skip): ')

os.environ['OPENROUTER_API_KEY'] = openrouter_key
if ngrok_token:
    os.environ['NGROK_TOKEN'] = ngrok_token

print('Keys set successfully' if openrouter_key else 'Warning: No OpenRouter API key set')

In [ ]:
# Start CoLM server and ngrok tunnel
import subprocess
import threading
import time

# Start the Node.js server
server_proc = subprocess.Popen(
    ['node', 'bin/cli.js'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ}
)

# Wait for server to start
time.sleep(2)

if ngrok_token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(5050, bind_tls=True)
    print(f'\nCoLM is running at: {tunnel.public_url}')
else:
    print('\nNo ngrok token — CoLM is running on localhost only')

print('\nServer logs (Ctrl+C to stop):')
print('='*50)

# Stream server logs
try:
    for line in iter(server_proc.stdout.readline, ''):
        print(line, end='')
except KeyboardInterrupt:
    server_proc.terminate()
    if ngrok_token:
        ngrok.disconnect(tunnel.public_url)
    print('\nCoLM stopped.')